In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

print("✅ STEP 1 COMPLETE — Environment Ready")

✅ STEP 1 COMPLETE — Environment Ready


In [2]:
import os

print("Available datasets:")
print(os.listdir('/kaggle/input'))

Available datasets:
['datasets', 'competitions']


In [3]:
print("Competition data:")
print(os.listdir('/kaggle/input/competitions'))

print("\nDataset data:")
print(os.listdir('/kaggle/input/datasets'))

Competition data:
['playground-series-s6e3']

Dataset data:
['blastchar']


In [4]:
print(os.listdir('/kaggle/input/datasets/blastchar'))

['telco-customer-churn']


In [5]:
print(os.listdir('/kaggle/input/datasets/blastchar/telco-customer-churn'))

['WA_Fn-UseC_-Telco-Customer-Churn.csv']


In [6]:
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
ORIG_PATH  = "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"

In [7]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
original = pd.read_csv(ORIG_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Original shape:", original.shape)

Train shape: (594194, 21)
Test shape: (254655, 20)
Original shape: (7043, 21)


In [8]:
original["Churn"] = (original["Churn"] == "Yes").astype(int)
original["TotalCharges"] = pd.to_numeric(original["TotalCharges"], errors="coerce")

if "id" not in original.columns:
    original["id"] = range(len(train), len(train) + len(original))

original = original[train.columns]
print("Original cleaned successfully")

Original cleaned successfully


In [9]:
cols = [c for c in train.columns if c not in ["id", "customerID"]]
train_hash = train[cols].apply(lambda row: hash(tuple(row)), axis=1)
orig_hash  = original[cols].apply(lambda row: hash(tuple(row)), axis=1)

is_duplicate = orig_hash.isin(train_hash)
print("Number of duplicates found:", is_duplicate.sum())
original_clean = original[~is_duplicate].reset_index(drop=True)

Number of duplicates found: 0


In [10]:
train_full = pd.concat([train, original], ignore_index=True)

print("Final train shape:", train_full.shape)
print("Test shape:", test.shape)

Final train shape: (601237, 21)
Test shape: (254655, 20)


In [11]:
def preprocess(df):
    df = df.copy()
    df["gender"] = (df["gender"] == "Male").astype(int)
    binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]
    for col in binary_cols:
        if col in df.columns:
            df[col] = (df[col] == "Yes").astype(int)
    df["Contract"] = df["Contract"].map({
        "Month-to-month": 0,
        "One year": 1,
        "Two year": 2
    })
    
    df["InternetService"] = df["InternetService"].map({
        "No": 0,
        "DSL": 1,
        "Fiber optic": 2
    })
    
    df["PaymentMethod"] = df["PaymentMethod"].map({
        "Electronic check": 0,
        "Mailed check": 1,
        "Bank transfer (automatic)": 2,
        "Credit card (automatic)": 3
    })
    service_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies", "MultipleLines"
    ]
    for col in service_cols:
        if col in df.columns:
            df[col] = (df[col] == "Yes").astype(int)

    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(df["tenure"] * df["MonthlyCharges"])
    return df

In [12]:
train_full = preprocess(train_full)
test = preprocess(test)

print("Preprocessing done")

Preprocessing done


In [13]:
def add_features(df):
    df = df.copy()
    df["charges_per_month"] = df["TotalCharges"] / (df["tenure"] + 1)
    service_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies", "MultipleLines"
    ]
    df["service_count"] = df[service_cols].sum(axis=1)
    df["is_new_customer"] = (df["tenure"] <= 3).astype(int)
    df["is_loyal_customer"] = (df["tenure"] >= 24).astype(int)

    df["log_monthly_charges"] = np.log1p(df["MonthlyCharges"])
    df["log_total_charges"] = np.log1p(df["TotalCharges"])

    df["charge_diff"] = df["MonthlyCharges"] - df["charges_per_month"]

    df["tenure_bin"] = pd.cut(
        df["tenure"],
        bins=[-1, 3, 12, 24, 48, 72, 999],
        labels=[0, 1, 2, 3, 4, 5]
    ).astype(float)
    
    df["monthly_charge_rank"] = df["MonthlyCharges"].rank(pct=True)
    df["auto_pay"] = (df["PaymentMethod"] >= 2).astype(int)
    df["no_internet_services"] = (df["InternetService"] == 0).astype(int)
    
    return df

In [14]:
train_full = add_features(train_full)
test = add_features(test)

print("Advanced features added")

Advanced features added


In [15]:
print("Total features:", len(train_full.columns))

Total features: 32


In [16]:
TARGET = "Churn"
drop_cols = ["id", "customerID"]

FEATURES = [col for col in train_full.columns if col not in drop_cols + [TARGET]]

print("Number of features used:", len(FEATURES))

Number of features used: 30


In [17]:
train_full["Churn"] = train_full["Churn"].replace({
    "Yes": 1,
    "No": 0
}).astype(int)

print("Unique values in target:", train_full["Churn"].unique())

Unique values in target: [0 1]


In [18]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

FOLDS = list(skf.split(train_full, train_full["Churn"]))

In [19]:
for i, (tr, val) in enumerate(FOLDS):
    tr_mean = train_full.iloc[tr]["Churn"].mean()
    val_mean = train_full.iloc[val]["Churn"].mean()
    
    print(f"Fold {i}: train={tr_mean:.4f}, val={val_mean:.4f}")

Fold 0: train=0.2257, val=0.2257
Fold 1: train=0.2257, val=0.2257
Fold 2: train=0.2257, val=0.2257
Fold 3: train=0.2257, val=0.2257
Fold 4: train=0.2257, val=0.2257


In [20]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

def train_lgb():
    oof = np.zeros(len(train_full))
    test_preds = np.zeros(len(test))
    
    for fold, (tr_idx, val_idx) in enumerate(FOLDS):
        X_tr = train_full.iloc[tr_idx][FEATURES]
        y_tr = train_full.iloc[tr_idx][TARGET]
        
        X_val = train_full.iloc[val_idx][FEATURES]
        y_val = train_full.iloc[val_idx][TARGET]
        
        model = lgb.LGBMClassifier(
            n_estimators=2000,
            learning_rate=0.01,
            num_leaves=16,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        
        oof[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(test[FEATURES])[:, 1] / 5
        
        print(f"Fold {fold} done")
    
    auc = roc_auc_score(train_full[TARGET], oof)
    print("LGB CV AUC:", auc)
    
    return oof, test_preds

In [21]:
lgb_oof, lgb_test = train_lgb()

[LightGBM] [Info] Number of positive: 108549, number of negative: 372440
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040957 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1915
[LightGBM] [Info] Number of data points in the train set: 480989, number of used features: 30


[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225679 -> initscore=-1.232874
[LightGBM] [Info] Start training from score -1.232874


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


Fold 0 done


In [ ]:
train_only = train_full.iloc[:len(train)].copy()
print("Train_only shape:", train_only.shape)

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

FOLDS = list(skf.split(train_only, train_only[TARGET]))

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

def train_lgb():
    oof = np.zeros(len(train_only))
    test_preds = np.zeros(len(test))
    
    for fold, (tr_idx, val_idx) in enumerate(FOLDS):
        
        X_tr = train_only.iloc[tr_idx][FEATURES]
        y_tr = train_only.iloc[tr_idx][TARGET]
        
        X_val = train_only.iloc[val_idx][FEATURES]
        y_val = train_only.iloc[val_idx][TARGET]
        
        model = lgb.LGBMClassifier(
            n_estimators=2000,
            learning_rate=0.01,
            num_leaves=16,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        
        oof[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(test[FEATURES])[:, 1] / 5
        
        print(f"Fold {fold} done")
    
    auc = roc_auc_score(train_only[TARGET], oof)
    print("LGB CV AUC:", auc)
    
    return oof, test_preds

In [ ]:
lgb_oof, lgb_test = train_lgb()

In [ ]:
train_only = train.copy()
test_only = test.copy()

In [ ]:
train_only = preprocess(train_only)
test_only = preprocess(test_only)

train_only = add_features(train_only)
test_only = add_features(test_only)

In [ ]:
train_only["Churn"] = train_only["Churn"].replace({
    "Yes": 1,
    "No": 0
})

train_only["Churn"] = train_only["Churn"].astype(int)
print("Unique target values:", train_only["Churn"].unique())

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

FOLDS = list(skf.split(train_only, train_only["Churn"]))

In [ ]:
lgb_oof, lgb_test = train_lgb()

In [ ]:
train_only = train.copy()
test_only = test.copy()

train_only["Churn"] = train_only["Churn"].replace({
    "Yes": 1,
    "No": 0
}).astype(int)

train_only = preprocess(train_only)
test_only = preprocess(test_only)

train_only = add_features(train_only)
test_only = add_features(test_only)

TARGET = "Churn"
drop_cols = ["id", "customerID"]

FEATURES = [c for c in train_only.columns if c not in drop_cols + [TARGET]]

print("Features:", len(FEATURES))

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

FOLDS = list(skf.split(train_only, train_only[TARGET]))

In [ ]:
lgb_oof, lgb_test = train_lgb()

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

def train_xgb():
    oof = np.zeros(len(train_only))
    test_preds = np.zeros(len(test_only))
    
    for fold, (tr_idx, val_idx) in enumerate(FOLDS):
        
        X_tr = train_only.iloc[tr_idx][FEATURES]
        y_tr = train_only.iloc[tr_idx][TARGET]
        
        X_val = train_only.iloc[val_idx][FEATURES]
        y_val = train_only.iloc[val_idx][TARGET]
        
        model = xgb.XGBClassifier(
            n_estimators=2000,
            learning_rate=0.01,
            max_depth=1,  # important
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="auc",
            random_state=42,
            verbosity=0
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        oof[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(test_only[FEATURES])[:, 1] / 5
        
        print(f"XGB Fold {fold} done")
    
    auc = roc_auc_score(train_only[TARGET], oof)
    print("XGB CV AUC:", auc)
    
    return oof, test_preds

In [ ]:
xgb_oof, xgb_test = train_xgb()

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

def train_cat():
    oof = np.zeros(len(train_only))
    test_preds = np.zeros(len(test_only))
    
    for fold, (tr_idx, val_idx) in enumerate(FOLDS):
        
        X_tr = train_only.iloc[tr_idx][FEATURES]
        y_tr = train_only.iloc[tr_idx][TARGET]
        
        X_val = train_only.iloc[val_idx][FEATURES]
        y_val = train_only.iloc[val_idx][TARGET]
        
        model = CatBoostClassifier(
            iterations=2000,
            learning_rate=0.01,
            depth=3,
            eval_metric="AUC",
            random_seed=42,
            verbose=False
        )
        
        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            verbose=False
        )
        
        oof[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(test_only[FEATURES])[:, 1] / 5
        
        print(f"CAT Fold {fold} done")
    
    auc = roc_auc_score(train_only[TARGET], oof)
    print("CAT CV AUC:", auc)
    
    return oof, test_preds

In [ ]:
cat_oof, cat_test = train_cat()

In [ ]:
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

def rank_avg(preds):
    return rankdata(preds) / len(preds)

# Rank averaging (important for stability)
final_oof = (
    rank_avg(lgb_oof) +
    rank_avg(xgb_oof) +
    rank_avg(cat_oof)
) / 3

final_test = (
    rank_avg(lgb_test) +
    rank_avg(xgb_test) +
    rank_avg(cat_test)
) / 3

print("Ensemble AUC:", roc_auc_score(train_only[TARGET], final_oof))

In [ ]:
final_oof = (
    0.5 * rank_avg(lgb_oof) +
    0.3 * rank_avg(cat_oof) +
    0.2 * rank_avg(xgb_oof)
)

final_test = (
    0.5 * rank_avg(lgb_test) +
    0.3 * rank_avg(cat_test) +
    0.2 * rank_avg(xgb_test)
)

print("Weighted Ensemble AUC:", roc_auc_score(train_only[TARGET], final_oof))

In [ ]:
stack_train = pd.DataFrame({
    "lgb": lgb_oof,
    "xgb": xgb_oof,
    "cat": cat_oof
})

stack_test = pd.DataFrame({
    "lgb": lgb_test,
    "xgb": xgb_test,
    "cat": cat_test
})

print(stack_train.head())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

meta_model = LogisticRegression()

meta_model.fit(stack_train, train_only[TARGET])

stack_oof = meta_model.predict_proba(stack_train)[:, 1]
stack_test_preds = meta_model.predict_proba(stack_test)[:, 1]

print("Stacking AUC:", roc_auc_score(train_only[TARGET], stack_oof))

In [ ]:
import lightgbm as lgb

meta_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=8,
    random_state=42
)

meta_model.fit(stack_train, train_only[TARGET])

stack_oof = meta_model.predict_proba(stack_train)[:, 1]
stack_test_preds = meta_model.predict_proba(stack_test)[:, 1]

print("Stacking AUC (LGB):", roc_auc_score(train_only[TARGET], stack_oof))

In [ ]:
high_conf_mask = (stack_test_preds > 0.99) | (stack_test_preds < 0.01)

pseudo_test = test_only[high_conf_mask].copy()
pseudo_labels = (stack_test_preds[high_conf_mask] > 0.5).astype(int)

print("Pseudo samples:", len(pseudo_test))

In [ ]:
pseudo_test["Churn"] = pseudo_labels

train_pseudo = pd.concat([train_only, pseudo_test], ignore_index=True)

print("New train size:", train_pseudo.shape)

In [ ]:
def train_lgb_pseudo():
    oof = np.zeros(len(train_only))
    test_preds = np.zeros(len(test_only))
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(train_only, train_only[TARGET])):
        
        X_tr = train_pseudo.iloc[tr_idx][FEATURES]
        y_tr = train_pseudo.iloc[tr_idx][TARGET]
        
        X_val = train_only.iloc[val_idx][FEATURES]
        y_val = train_only.iloc[val_idx][TARGET]
        
        model = lgb.LGBMClassifier(
            n_estimators=2000,
            learning_rate=0.01,
            num_leaves=16,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        
        oof[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(test_only[FEATURES])[:, 1] / 5
        
        print(f"Pseudo Fold {fold} done")
    
    auc = roc_auc_score(train_only[TARGET], oof)
    print("Pseudo LGB AUC:", auc)
    
    return oof, test_preds

In [ ]:
pseudo_oof, pseudo_test_preds = train_lgb_pseudo()